## Samodzielna analiza skupień - produkty spożywcze

W tym notebooku wykorzystasz schemat analizy pokazany wcześniej na zbiorze Iris, ale zastosujesz go do innego, bardziej praktycznego zbioru danych.

Zbiór zawiera produkty spożywcze opisane przez zawartość różnych składników odżywczych, takich jak białko, tłuszcz, węglowodany, składniki mineralne i witaminy.

Celem analizy będzie pogrupowanie produktów spożywczych na podstawie ich profilu odżywczego.

W przeciwieństwie do zbioru Iris nie mamy tutaj zmiennej referencyjnej typu `species`.  
Nie znamy więc prawdziwego podziału produktów na grupy. Jakość segmentacji będziemy oceniać na podstawie miar wewnętrznych, liczności skupień, profili skupień oraz interpretowalności wyniku.

W pierwszej części notebooka przygotujemy dane do analizy:

- zaimportujemy dane,
- rozpoznamy strukturę zbioru,
- ocenimy braki danych,
- wykonamy podstawowe czyszczenie danych,
- sprawdzimy rozkłady i zależności między zmiennymi,
- przygotujemy dane do samodzielnej analizy skupień.

In [ ]:
# Dodatkowe ułatwienia w ustawieniach wyświetlania i obsłudze ostrzeżeń

from IPython.display import display, HTML
import warnings
import os

# Wyłączenie przewijanej ramki wyników w Jupyterze
display(HTML("<style>.output_scroll { height: auto !important; }</style>"))

# Wyłączenie wybranych ostrzeżeń
warnings.simplefilter(action="ignore", category=UserWarning)

# Wyświetlenie aktualnego katalogu roboczego
print(os.getcwd())

### Poniżej lista często wykorzystywanych bibliotek, ich rolę będziemy poznawali krok po kroku

In [ ]:
# Struktury danych i podstawowe operacje
import pandas as pd
import numpy as np

# Wykresy
import matplotlib.pyplot as plt
import seaborn as sns

# Uzupełnianie braków danych
from sklearn.impute import SimpleImputer, KNNImputer

# Obsługa wartości odstających
from feature_engine.outliers import Winsorizer

# Skalowanie zmiennych
from sklearn.preprocessing import MinMaxScaler

# Wartość początkowa dla generatora liczb losowych
seed = 1

### Dodatkowe biblioteki potrzebne do analizy

In [ ]:
# Jeśli biblioteka MiniSom nie jest zainstalowana, można ją doinstalować
#!pip install minisom

In [ ]:
import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture
from minisom import MiniSom
from sklearn.cluster import DBSCAN

## Co zawiera zbiór i po co robimy analizę?

Zbiór zawiera produkty spożywcze opisane przez zawartość różnych składników odżywczych, m.in. białka, tłuszczu, węglowodanów, składników mineralnych i witamin.

Celem analizy jest pogrupowanie produktów spożywczych w zależności od ich profilu odżywczego.

Wynik analizy powinien pozwolić odpowiedzieć na pytania:

- czy produkty tworzą naturalne grupy pod względem składu,
- czym różnią się otrzymane skupienia,
- które składniki najbardziej charakteryzują poszczególne grupy,
- czy skupienia dają się sensownie nazwać i zinterpretować.

### Import danych i pierwsze pytania do danych

1. Jaka jest struktura danych? 
2. Ile jest przypadków? Ile wierszy?
3. Co one oznaczają?
4. Jakie są typy kolumn w ramce danych?
5. Jaka jest liczba różnych wartości?

In [ ]:
# Import danych z pliku xlsx do ramki danych food_df
# Plik powinien znajdować się w tym samym folderze co notebook.

data_path = "skladniki_simple.xlsx"

food_df = pd.read_excel(
    data_path,
    header=1,
    index_col=0
)

food_df.head()

In [ ]:
# Jaka jest liczba przypadków? Jaka jest liczba zmiennych?
print(food_df.shape)

In [ ]:
# Pokazuje wszystkie kolumny zbioru podczas podglądu
pd.set_option("display.max_columns", None) 

# Wyświetlenie kilku pierwszych wierszy
food_df.head(10)

In [ ]:
# Podstawowe sprawdzenie typów danych
food_df.dtypes

In [ ]:
# Sprawdzenie liczby unikalnych wartości
food_df.nunique()

## Pierwsza decyzja: rola zmiennych w analizie

Do analizy skupień wykorzystamy zmienne liczbowe opisujące skład produktów.

Kolumna identyfikująca produkt nie powinna być traktowana jako zmienna analityczna.  
Jeżeli identyfikator produktu został wczytany jako indeks ramki danych, nie trzeba go dodatkowo usuwać z listy zmiennych.

In [ ]:
# Określ rolę zmiennych w analizie 
id_variable = ['ID']
predictors = [var for var in food_df.columns if var not in id_variable]

In [ ]:
print(predictors)

## Brak zmiennej referencyjnej

W tym zbiorze nie mamy zmiennej, która mówiłaby, do jakiej „prawdziwej” grupy należy dany produkt.

Jest to typowa sytuacja w analizie skupień.  
Nie możemy więc ocenić modelu przez porównanie z rzeczywistymi etykietami, tak jak robiliśmy to w zbiorze Iris za pomocą zmiennej `species`.

W tej analizie jakość skupień będziemy oceniać na podstawie:

- miar wewnętrznych, takich jak wskaźnik sylwetki,
- liczności skupień,
- profili skupień,
- interpretowalności otrzymanych grup.

## Ocena jakości zbioru danych

Przed modelowaniem sprawdzimy jakość danych.

Na tym etapie ocenimy:

- braki danych w zmiennych,
- braki danych w obrębie obserwacji,
- podstawowe rozkłady zmiennych ilościowych,
- potencjalne wartości odstające,
- zmienne, które mogą wymagać imputacji lub usunięcia.

W tym zbiorze szczególnie ważna będzie analiza braków danych, ponieważ niektóre produkty mogą mieć niepełny opis składu.

In [ ]:
# Raport braków danych dla zmiennych używanych w analizie

missing_report = pd.DataFrame({
    "Liczba braków": food_df[predictors].isnull().sum(),
    "Procent braków": (food_df[predictors].isnull().mean() * 100).round(2)
})

missing_report.sort_values("Procent braków", ascending=False)

W przeciwieństwie do zbioru Iris, tutaj występują braki danych.

Zanim zdecydujemy, jak je uzupełnić, warto sprawdzić nie tylko braki w poszczególnych zmiennych, ale także to, czy niektóre produkty mają szczególnie dużo brakujących informacji.

In [ ]:
# Analiza braków danych w obrębie obserwacji

null_row_report = pd.DataFrame({
    "Odsetek braków": food_df[predictors].isnull().mean(axis=1)
})

null_row_report = null_row_report.sort_values(
    "Odsetek braków",
    ascending=False
)

null_row_report.head(10)

## Pierwsza ingerencja w dane

Część produktów może mieć bardzo dużo brakujących wartości.  
Takie obserwacje mogą być trudne do wiarygodnego uzupełnienia i mogą zaburzać wynik analizy skupień.

W tym przykładzie usuniemy z analizy produkty, dla których brakuje więcej niż 40% wartości zmiennych używanych w analizie.

Jest to decyzja analityczna - w innych projektach próg ten może być inny i powinien zależeć od kontekstu danych.

### Pierwsza ingerencja w dane

In [ ]:
# Zachowujemy kopię danych przed usunięciem obserwacji
food_df_before_row_filtering = food_df.copy()

# Usuwamy z analizy obserwacje, dla których odsetek braków danych przekracza 40%
row_missing_share = food_df[predictors].isnull().mean(axis=1)
food_df = food_df.loc[row_missing_share <= 0.4].copy()

print(food_df.shape)

In [ ]:
# Ponowny raport braków danych po usunięciu obserwacji z dużą liczbą braków

missing_report = pd.DataFrame({
    "Liczba braków": food_df[predictors].isnull().sum(),
    "Procent braków": (food_df[predictors].isnull().mean() * 100).round(2)
})

missing_report.sort_values("Procent braków", ascending=False)

## Wybór techniki uzupełniania braków danych

Po usunięciu obserwacji z bardzo dużą liczbą braków nadal mogą pozostać pojedyncze braki w zmiennych.

Sposób uzupełniania braków zależy między innymi od ich liczby:

- dla zmiennych z niewielkim odsetkiem braków można zastosować prostą imputację medianą,
- dla zmiennych z większym odsetkiem braków można rozważyć bardziej zaawansowaną metodę, np. imputację KNN.

W tym przykładzie przyjmiemy próg 5% braków danych.

In [ ]:
# Podział zmiennych z brakami danych według odsetka braków

missing_over_5 = [
    var for var in predictors
    if food_df[var].isnull().mean() >= 0.05
]

missing_below_5 = [
    var for var in predictors
    if 0 < food_df[var].isnull().mean() < 0.05
]

print("Zmienne z odsetkiem braków >= 5%:")
print(missing_over_5)

print("\nZmienne z odsetkiem braków < 5%:")
print(missing_below_5)

## Badanie rozkładów zmiennych

Po analizie braków danych sprawdzimy rozkłady zmiennych ilościowych.

Na tym etapie interesuje nas:

- liczba unikalnych wartości,
- podstawowe statystyki opisowe,
- kształt rozkładów,
- potencjalne wartości odstające,
- zmienne o bardzo nietypowym zakresie wartości.

W tym zbiorze do analizy skupień wykorzystujemy zmienne ilościowe opisujące skład produktów.  
Nie mamy tutaj zmiennych jakościowych, dlatego nie wykonujemy osobnej analizy tabel liczności dla kategorii.

In [ ]:
# Liczba unikalnych wartości w zmiennych używanych w analizie

food_df[predictors].nunique().sort_values()

In [ ]:
# Podsumowanie podstawowych statystyk opisowych
food_df[predictors].describe().T

In [ ]:
# Histogramy dla zmiennych ilościowych

food_df[predictors].hist(
    bins=20,
    figsize=(15, 15)
)

plt.suptitle("Rozkłady zmiennych ilościowych", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def plot_box_strip(df, variables, hue=None):
    """
    Tworzy wykresy pudełkowe z nałożonymi punktami obserwacji dla podanych zmiennych.

    Parametry:
    - df: DataFrame zawierający dane,
    - variables: lista nazw kolumn ze zmiennymi ilościowymi,
    - hue: opcjonalna zmienna grupująca.
    """

    for var in variables:
        plt.figure(figsize=(8, 6))

        sns.boxplot(
            data=df,
            y=var,
            x=hue,
            fliersize=0,
            color="green"
        )

        sns.stripplot(
            data=df,
            y=var,
            x=hue,
            jitter=0.1,
            alpha=0.3,
            color="black"
        )

        plt.title(f"Wykres pudełkowy z obserwacjami: {var}")
        plt.xlabel(hue if hue else "")
        plt.ylabel(var)
        plt.tight_layout()
        plt.show()

In [ ]:
plot_box_strip(food_df, predictors)

Wykresy histogramów i wykresy pudełkowe pomagają ocenić, czy zmienne mają rozkłady symetryczne, skośne lub zawierają wartości nietypowe.

W analizie produktów spożywczych wartości odstające nie muszą oznaczać błędów.  
Produkt o bardzo wysokiej zawartości tłuszczu, błonnika albo określonego mikroelementu może być po prostu specyficznym produktem, a nie błędną obserwacją.

Dlatego wartości odstające należy najpierw zidentyfikować, a dopiero później zdecydować, czy wymagają przycięcia, usunięcia albo pozostawienia w analizie.

## Czyszczenie danych: uzupełnianie braków

Po rozpoznaniu danych przechodzimy do ich przygotowania do analizy skupień.

W tym zbiorze występują braki danych, dlatego musimy zdecydować, jak je uzupełnić.

Przyjmujemy następującą strategię:

- zmienne z niewielkim odsetkiem braków uzupełnimy medianą,
- zmienne z większym odsetkiem braków uzupełnimy metodą KNN.

Warto pamiętać, że wybór metody imputacji jest decyzją analityczną.  
W innych projektach sposób uzupełniania braków powinien zależeć od charakteru danych, skali zmiennych oraz celu analizy.

In [ ]:
# Uzupełnianie braków danych

# Dla zmiennych z niewielkim odsetkiem braków stosujemy medianę
if len(missing_below_5) > 0:
    simple_imputer = SimpleImputer(strategy="median")
    food_df.loc[:, missing_below_5] = simple_imputer.fit_transform(
        food_df[missing_below_5]
    )

# Dla pozostałych braków stosujemy imputację KNN
# Uwaga: KNN opiera się na odległościach, więc w bardziej rygorystycznej analizie
# warto rozważyć wcześniejsze przeskalowanie zmiennych.

knn_imputer = KNNImputer(
    n_neighbors=3,
    weights="distance"
)

food_df.loc[:, predictors] = knn_imputer.fit_transform(food_df[predictors])

# Sprawdzenie, czy braki zostały uzupełnione

null_report_after_imputation = food_df[predictors].isnull().mean().sort_values(
    ascending=False
)

null_report_after_imputation

> **Uwaga metodyczna**
>
> Imputacja KNN opiera się na podobieństwie obserwacji, które jest wyznaczane na podstawie odległości między nimi.
>
> Oznacza to, że zmienne o większych wartościach lub szerszym zakresie mogą mieć silniejszy wpływ na wynik uzupełniania braków danych.
>
> W tym przykładzie pozostawiamy procedurę w uproszczonej formie, ale w bardziej rygorystycznej analizie warto rozważyć przeskalowanie zmiennych przed zastosowaniem imputacji KNN.

## Czyszczenie danych: wartości odstające

Kolejnym krokiem jest ocena wartości odstających.

Wartości odstające nie zawsze są błędami. W przypadku produktów spożywczych mogą oznaczać produkty rzeczywiście nietypowe, np. bardzo tłuste, bardzo bogate w białko albo zawierające wyjątkowo dużo wybranego składnika.

W tym przykładzie zastosujemy winsoryzację, czyli przycięcie wartości skrajnych do granic wyznaczonych metodą IQR.  
Traktujemy to jako jedną z możliwych metod ograniczania wpływu wartości ekstremalnych na analizę skupień.

In [ ]:
# Zachowujemy kopię danych przed winsoryzacją

food_df_before_winsorization = food_df.copy()

# capping_method='iqr' - metoda oparta na rozstępie kwartylowym
# fold=1.5 - parametr sterujący szerokością granic przycinania
# tail='both' - przycinanie wartości odstających z obu stron rozkładu

winsorizer = Winsorizer(
    capping_method="iqr",
    fold=1.5,
    tail="both"
)

food_df.loc[:, predictors] = winsorizer.fit_transform(food_df[predictors])

In [ ]:
# Sprawdzenie efektu winsoryzacji
plot_box_strip(food_df, predictors)

## Powiązania pomiędzy zmiennymi

Przed skalowaniem i modelowaniem warto sprawdzić zależności pomiędzy zmiennymi.

Silnie skorelowane zmienne mogą wnosić podobną informację do modelu.  
Nie oznacza to automatycznie, że którąś z nich trzeba usunąć, ale może być podstawą do rozważenia uproszczenia zestawu zmiennych.

W tym celu obliczymy korelacje Spearmana i przedstawimy je na mapie ciepła.

In [ ]:
# Obliczanie macierzy korelacji rang Spearmana

spearman_corr_matrix = food_df[predictors].corr(method="spearman")

spearman_corr_matrix

In [ ]:
# Mapa ciepła korelacji

plt.figure(figsize=(12, 10))

sns.heatmap(
    spearman_corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Mapa ciepła korelacji rang Spearmana")
plt.tight_layout()
plt.show()

Zmienne `Tłuszcz (g)` oraz `Energia (kcal)` są silnie powiązane.

Możemy rozważyć usunięcie jednej z nich, aby ograniczyć powielanie podobnej informacji w analizie.  
W tym przykładzie usuniemy zmienną `Energia (kcal)`, ponieważ jest ona w dużej mierze pochodną składu odżywczego produktu.

In [ ]:
# Usunięcie zmiennej Energia (kcal) z listy predyktorów

predictors = [var for var in predictors if var != "Energia (kcal)"]

print(predictors)

In [ ]:
def scatter_plot(df, variables, hue=None):
    """
    Tworzy wykres rozrzutu dla dwóch podanych zmiennych.

    Parametry:
    - df: DataFrame z danymi,
    - variables: lista dokładnie 2 nazw kolumn [x, y],
    - hue: opcjonalna zmienna jakościowa do kolorowania punktów.
    """

    if len(variables) != 2:
        raise ValueError("Lista 'variables' musi zawierać dokładnie 2 zmienne.")

    x, y = variables

    plt.figure(figsize=(8, 6))

    if hue:
        sns.scatterplot(data=df, x=x, y=y, hue=hue, palette="Set1", alpha=0.7)
        plt.legend(title=hue)
    else:
        sns.scatterplot(data=df, x=x, y=y, alpha=0.7)

    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f"Wykres rozrzutu: {x} vs {y}")
    plt.tight_layout()
    plt.show()

In [ ]:
print(predictors)

In [ ]:
scatter_plot(food_df, ["Białko (g)", "Tłuszcz (g)"])

## Skalowanie danych

Algorytmy analizy skupień często wykorzystują odległości między obserwacjami.

Jeżeli zmienne mają różne skale, zmienne o większych wartościach mogą silniej wpływać na wynik grupowania.  
Dlatego przed modelowaniem przeskalujemy dane do wspólnego zakresu.

W tym przykładzie użyjemy `MinMaxScaler`, który przekształca wartości zmiennych do przedziału od 0 do 1.

In [ ]:
# Skalowanie wartości predyktorów
# Przed wykonaniem skalowania tworzymy kopię ramki danych

scaled_food_df = food_df.copy()

min_max_scaler = MinMaxScaler()

scaled_food_df.loc[:, predictors] = min_max_scaler.fit_transform(
    scaled_food_df[predictors]
)

scaled_food_df[predictors].head()

In [ ]:
# Sprawdzenie rozkładów zmiennych po skalowaniu
plot_box_strip(scaled_food_df, predictors)

# Zadanie 3 -  Samodzielna analiza skupień produktów spożywczych

Dane zostały przygotowane do analizy skupień.

Na podstawie przygotowanych ramek:

- `food_df`,
- `scaled_food_df`,
- `predictors`

wykonaj samodzielną analizę skupień produktów spożywczych.

Wykorzystaj notebook ze zbiorem Iris jako wzorzec postępowania.

## Co należy zrobić?

1. Dobierz liczbę skupień, korzystając z co najmniej dwóch kryteriów, np.:
   - wykresu osypiska,
   - wskaźnika sylwetki,
   - Calinski-Harabasz Score,
   - Davies-Bouldin Score.

2. Zbuduj model analizy skupień wybraną metodą, np. KMeans lub metodą aglomeracyjną.

3. Oceń jakość otrzymanego podziału:
   - sprawdź liczności skupień,
   - oblicz wybrane miary jakości,
   - przygotuj wizualizację skupień za pomocą PCA lub t-SNE.

4. Zinterpretuj otrzymane skupienia:
   - czym różnią się grupy produktów,
   - które zmienne najlepiej opisują poszczególne skupienia,
   - jak można nazwać otrzymane grupy.

## Efekt końcowy

Przygotuj krótkie podsumowanie analizy, w którym wskażesz:

- wybraną metodę,
- wybraną liczbę skupień,
- najważniejsze wyniki oceny jakości,
- interpretację otrzymanych grup produktów.

# Zadanie 4 - Analiza skupień po redukcji wymiarów

W przygotowanym notebooku jedna z decyzji analitycznych polegała na usunięciu zmiennej `Energia (kcal)`, ponieważ była silnie powiązana z innymi zmiennymi, zwłaszcza z zawartością tłuszczu.

W tym zadaniu sprawdzisz inne podejście: zamiast ręcznie usuwać silnie powiązane zmienne, zastosujesz redukcję wymiarów, a następnie wykonasz analizę skupień na uzyskanych składowych lub czynnikach.

## Co należy zrobić?

1. Przygotuj dane liczbowe do analizy.
2. Nie usuwaj zmiennej `Energia (kcal)` na podstawie samej korelacji.
3. Sprawdź, czy struktura korelacji między zmiennymi uzasadnia zastosowanie redukcji wymiarów:
   - oblicz miarę KMO,
   - wykonaj test sferyczności Bartletta.
4. Wykonaj PCA.
5. Wybierz liczbę składowych na podstawie:
   - wartości własnych,
   - procentu wyjaśnionej wariancji,
   - wykresu osypiska,
   - interpretowalności składowych.
6. Zapisz wartości składowych dla obserwacji.
7. Wykonaj analizę skupień na uzyskanych składowych.
8. Porównaj wyniki z analizą skupień wykonaną na oryginalnych zmiennych.

## Co należy porównać?

Porównaj oba podejścia pod względem:

- liczby i liczności skupień,
- wartości wskaźnika sylwetki,
- wartości Calinski-Harabasz Score,
- wartości Davies-Bouldin Score,
- interpretowalności profili skupień,
- zgodności lub podobieństwa uzyskanych podziałów.

Do porównania dwóch różnych podziałów możesz wykorzystać Adjusted Rand Index, traktując jeden model jako punkt odniesienia.

## Pytania do interpretacji

1. Czy KMO i test Bartletta sugerują, że redukcja wymiarów ma sens?
2. Ile składowych lub czynników warto zachować?
3. Jak można zinterpretować uzyskane składowe/czynniki?
4. Czy analiza skupień na skorach daje bardziej czytelne grupy niż analiza na oryginalnych zmiennych?
5. Czy wyniki obu podejść są podobne?
6. Które podejście jest lepsze: analiza na oryginalnych zmiennych czy analiza po redukcji wymiarów?
7. Czy usunięcie zmiennej `Energia (kcal)` było konieczne, czy redukcja wymiarów lepiej rozwiązuje problem współzależności zmiennych?

## Uwaga metodyczna

KMO i test Bartletta służą do oceny, czy dane mają strukturę korelacyjną, która uzasadnia zastosowanie analizy PCA lub podobnych metod redukcji wymiarów.

Dopiero po redukcji wymiarów wykonujemy analizę skupień i oceniamy jej jakość za pomocą miar typowych dla segmentacji.